# [7.4] Mini Natural Language Autoencoders - Exercises

Build the local NLA evaluation contract: aligned text-bottleneck records, reconstruction metrics, behavioral preservation checks, compression controls, and counterfactual explanation checks. In the full CUDA path, a small encoder/decoder pair is trained over short natural-language phrase ids; the bottleneck is text, not signed numeric residual coordinates.

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part4_mini_natural_language_autoencoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_mini_natural_language_autoencoders.tests as tests

GT_TIER = "GT-3"
EXERCISE_ID = "7.4.mini_natural_language_autoencoders"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1 minute for CUDA preflight"
REQUIRES_GPU = False

## Text-Bottleneck Records

Keep every activation row aligned with its source text, synthetic latent label, and generated explanation.

In [ ]:
@dataclass(frozen=True)
class NLATrainingBatch:
    activations: t.Tensor
    original_text_spans: tuple[str, ...]
    synthetic_latent_labels: tuple[str, ...]
    generated_explanations: tuple[str, ...]


def build_nla_training_batch(
    activations: t.Tensor,
    original_text_spans: list[str],
    synthetic_latent_labels: list[str],
    generated_explanations: list[str],
) -> NLATrainingBatch:
    """Bundle activations with source spans, labels, and generated explanations."""
    raise NotImplementedError()


tests.test_build_nla_training_batch_validates_alignment(build_nla_training_batch)

## Reconstruction Quality

A reconstructed activation should be closer to the original activation than a text-only reconstruction baseline.

In [ ]:
@dataclass(frozen=True)
class NLAReconstructionReport:
    activation_mse: float
    text_only_mse: float
    mean_cosine_similarity: float
    beats_text_only: bool


def activation_reconstruction_report(
    original_activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    text_only_reconstructions: t.Tensor,
) -> NLAReconstructionReport:
    raise NotImplementedError()


tests.test_activation_reconstruction_report_beats_text_only_baseline(
    activation_reconstruction_report,
)

## Behavioral Preservation

Check the target positive-minus-negative logit difference and probe-decoded latent state, not just reconstruction MSE.

In [ ]:
@dataclass(frozen=True)
class LogitDiffPreservationReport:
    original_logit_diff: float
    reconstructed_logit_diff: float
    mean_abs_error: float
    preserves_target_logit_diff: bool


def batch_target_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> t.Tensor:
    raise NotImplementedError()


def logit_diff_preservation_report(
    original_logits: t.Tensor,
    reconstructed_logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
    max_mean_abs_error: float = 0.25,
) -> LogitDiffPreservationReport:
    raise NotImplementedError()


tests.test_logit_diff_preservation_report_checks_actual_logit_diff(
    logit_diff_preservation_report,
)

In [ ]:
@dataclass(frozen=True)
class LatentPreservationReport:
    original_probe_accuracy: float
    reconstructed_probe_accuracy: float
    prediction_agreement: float
    preserves_latents: bool


def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def latent_preservation_report(
    original_probe_logits: t.Tensor,
    reconstructed_probe_logits: t.Tensor,
    latent_ids: t.Tensor,
    *,
    min_accuracy: float = 0.75,
    min_agreement: float = 0.75,
) -> LatentPreservationReport:
    raise NotImplementedError()


tests.test_latent_preservation_report_requires_accuracy_and_agreement(
    latent_preservation_report,
)

## Compression And Counterfactuals

Reject prompt-copying explanations and unchanged counterfactual explanations.

In [ ]:
@dataclass(frozen=True)
class GeneratedTextBrevityReport:
    generated_word_count: int
    original_word_count: int
    compression_ratio: float
    shorter_than_original: bool


@dataclass(frozen=True)
class CounterfactualExplanationReport:
    original_explanation: str
    counterfactual_explanation: str
    activation_delta: float
    explanation_changed: bool


def generated_text_brevity_report(
    generated_explanations: list[str],
    original_prompts: list[str],
) -> GeneratedTextBrevityReport:
    raise NotImplementedError()


def counterfactual_explanation_report(
    original_activation: t.Tensor,
    counterfactual_activation: t.Tensor,
    original_explanation: str,
    counterfactual_explanation: str,
    *,
    min_activation_delta: float = 0.0,
) -> CounterfactualExplanationReport:
    raise NotImplementedError()


tests.test_brevity_and_counterfactual_reports_reject_prompt_copying(
    generated_text_brevity_report,
    counterfactual_explanation_report,
)

## Trainable Discrete Bottleneck

Train a tiny activation-to-phrase encoder and phrase-to-activation decoder. The bottleneck is a phrase id, not a residual-coordinate payload.


In [ ]:
@dataclass(frozen=True)
class TrainableNLABottleneckReport:
    encoder_final_loss: float
    decoder_final_mse: float
    encoder_train_accuracy: float
    eval_phrase_accuracy: float
    reconstruction_mse: float
    blank_text_mse: float
    beats_blank_text: bool
    generated_explanations: tuple[str, ...]
    phrase_count: int
    training_steps: int
    seed: int


def train_discrete_nla_bottleneck(
    train_activations: t.Tensor,
    train_phrase_ids: t.Tensor,
    eval_activations: t.Tensor,
    eval_phrase_ids: t.Tensor,
    phrase_texts: tuple[str, ...],
    *,
    steps: int = 300,
    lr: float = 0.05,
    seed: int = 0,
) -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor, t.Tensor, TrainableNLABottleneckReport]:
    raise NotImplementedError()


tests.test_trainable_discrete_bottleneck_learns_phrase_ids(
    train_discrete_nla_bottleneck,
)


## Combined Contract

After all helpers pass, compose them into a single CPU-only report. Then inspect the committed CUDA report: it should use a discrete phrase bottleneck, report `numeric_literal_count == 0`, beat text-only and prompt-label baselines, preserve latent predictions, and make shuffled-text plus blank-text controls worse than the generated explanations.

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
